# Data Preparation

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 14.01.2025 (Sebastian Birk; sebastian.birk@helmholtz-munich.de)

## 1. Setup

### 1.1 Import Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
import pprint
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyensembl
#import s3fs
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
import tarfile
#import zarr
from IPython.display import HTML, display
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

import nichejepa

### 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
cluster_colors = {
    "0": "#66C5CC",
    "1": "#F6CF71",
    "2": "#F89C74",
    "3": "#DCB0F2",
    "4": "#87C55F",
    "5": "#9EB9F3",
    "6": "#FE88B1",
    "7": "#C9DB74",
    "8": "#8BE0A4",
    "9": "#B497E7",
    "10": "#D3B484",
    "11": "#B3B3B3",
    "12": "#276A8C", # Royal Blue
    "13": "#DAB6C4", # Pink
    "14": "#C38D9E", # Mauve-Pink
    "15": "#9D88A2", # Mauve
    "16": "#FF4D4D", # Light Red
    "17": "#9B4DCA", # Lavender-Purple
    "18": "#FF9CDA", # Bright Pink
    "19": "#FF69B4", # Hot Pink
    "20": "#FF00FF", # Magenta
    "21": "#DA70D6", # Orchid
    "22": "#BA55D3", # Medium Orchid
    "23": "#8A2BE2", # Blue Violet
    "24": "#9370DB", # Medium Purple
    "25": "#7B68EE", # Medium Slate Blue
    "26": "#4169E1", # Royal Blue
    "27": "#FF8C8C", # Salmon Pink
    "28": "#FFAA80", # Light Coral
    "29": "#48D1CC", # Medium Turquoise
    "30": "#40E0D0", # Turquoise
    "31": "#00FF00", # Lime
    "32": "#7FFF00", # Chartreuse
    "33": "#ADFF2F", # Green Yellow
    "34": "#32CD32", # Lime Green
    "35": "#228B22", # Forest Green
    "36": "#FFD8B8", # Peach
    "37": "#008080", # Teal
    "38": "#20B2AA", # Light Sea Green
    "39": "#00FFFF", # Cyan
    "40": "#00BFFF", # Deep Sky Blue
    "41": "#4169E1", # Royal Blue
    "42": "#0000CD", # Medium Blue
    "43": "#00008B", # Dark Blue
    "44": "#8B008B", # Dark Magenta
    "45": "#FF1493", # Deep Pink
    "46": "#FF4500", # Orange Red
    "47": "#006400", # Dark Green
    "48": "#FF6347", # Tomato
    "49": "#FF7F50", # Coral
    "50": "#CD5C5C", # Indian Red
    "51": "#B22222", # Fire Brick
    "52": "#FFB83F",  # Light Orange
    "53": "#8B0000", # Dark Red
    "54": "#D2691E", # Chocolate
    "55": "#A0522D", # Sienna
    "56": "#800000", # Maroon
    "57": "#808080", # Gray
    "58": "#A9A9A9", # Dark Gray
    "59": "#C0C0C0", # Silver
    "60": "#9DD84A",
    "61": "#F5F5F5", # White Smoke
    "62": "#F17171", # Light Red
    "63": "#000000", # Black
    "64": "#FF8C42", # Tangerine
    "65": "#F9A11F", # Bright Orange-Yellow
    "66": "#FACC15", # Golden Yellow
    "67": "#E2E062", # Pale Lime
    "68": "#BADE92", # Soft Lime
    "69": "#70C1B3", # Greenish-Blue
    "70": "#41B3A3", # Turquoise
    "71": "#5EAAA8", # Gray-Green
    "72": "#72B01D", # Chartreuse
    "73": "#9CD08F", # Light Green
    "74": "#8EBA43", # Olive Green
    "75": "#FAC8C3", # Light Pink
    "76": "#E27D60", # Dark Salmon
    "77": "#C38D9E", # Mauve-Pink
    "78": "#937D64", # Light Brown
    "79": "#B1C1CC", # Light Blue-Gray
    "80": "#88A0A8", # Gray-Blue-Green
    "81": "#4E598C", # Dark Blue-Purple
    "82": "#4B4E6D", # Dark Gray-Blue
    "83": "#8E9AAF", # Light Blue-Grey
    "84": "#C0D6DF", # Pale Blue-Grey
    "85": "#97C1A9", # Blue-Green
    "86": "#4C6E5D", # Dark Green
    "87": "#95B9C7", # Pale Blue-Green
    "88": "#C1D5E0", # Pale Gray-Blue
    "89": "#ECDB54", # Bright Yellow
    "90": "#E89B3B", # Bright Orange
    "91": "#CE5A57", # Deep Red
    "92": "#C3525A", # Dark Red
    "93": "#B85D8E", # Berry
    "94": "#7D5295", # Deep Purple
    "-1" : "#E1D9D1",
    "None" : "#E1D9D1"
}

- Run ```pyensembl install --release 111 --species homo_sapiens```.
- Run Run ```pyensembl install --release 111 --species mus_musculus```.

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
st_data_folder_path = "/lustre/scratch126/cellgen/team361/DATASETS" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
st_data_url_folder_path = f"{st_data_folder_path}/urls"
sc_data_folder_path = "../datasets/sc_data" # single-cell data
sc_data_bronze_folder_path = f"{sc_data_folder_path}/bronze"
sc_data_silver_folder_path = f"{sc_data_folder_path}/silver"
sc_data_gold_folder_path = f"{sc_data_folder_path}/gold"

artifacts_folder_path = '../../artifacts'

# Create required directories
os.makedirs(st_data_bronze_folder_path, exist_ok=True)
os.makedirs(st_data_silver_folder_path, exist_ok=True)
os.makedirs(st_data_gold_folder_path, exist_ok=True)
os.makedirs(st_data_url_folder_path, exist_ok=True)
os.makedirs(sc_data_bronze_folder_path, exist_ok=True)
os.makedirs(sc_data_silver_folder_path, exist_ok=True)
os.makedirs(sc_data_gold_folder_path, exist_ok=True)

## 2. Manuscript Data

### 2.1 Nanostring CosMx Human Brain

- Data access: https://nanostring.com/products/cosmx-spatial-molecular-imager/ffpe-dataset/human-frontal-cortex-ffpe-dataset/

In [ ]:
dataset_name = "nanostring_cosmx_human_brain"
species = "homo_sapiens"

In [ ]:
ensembl = pyensembl.EnsemblRelease(111, species=species)

In [ ]:
# Read data
df = pd.read_csv(f"{st_data_bronze_folder_path}/{dataset_name}/S3_exprMat_file.csv.gz")
df_metadata = pd.read_csv(f"{st_data_bronze_folder_path}/{dataset_name}/S3_metadata_file.csv.gz")

#### 2.1.1 Preprocess Data

In [ ]:
## Create adata object
# Only keep genes
genes = [col for col in df.columns if
         ("FalseCode" not in col) &
         ("NegPrb" not in col) &
         ("Custom" not in col) &
         ("cell" not in col) &
         ("fov" not in col)]
df_counts = df[genes]

genes = [gene.split("/")[0] for gene in df_counts.columns]

# Rename some genes with names from Ensembl database
for old_gene_name, new_gene_name in zip(["BAGE", "LILRA3", "COX1", "COX2", "CT45A", "HLA-DRB", "LOC728392", "METTL7A", "SLC9A3R1", "TTC30A"],
                                        ["BAGE2", "ENSG00000275841", "MT-CO1", "MT-CO2", "CT45A1", "HLA-DRB1", "ENSG00000286190", "TMT1A", "NHERF1", "IFT70A"]):
    idx = genes.index(old_gene_name)
    genes[idx] = new_gene_name
    
df_counts.columns = genes

var_metadata = pd.DataFrame(index=df_counts.columns)
obs_metadata = pd.DataFrame(index=df_counts.index, data={"fov": df["fov"].values,
                                                         "cell_id": df["cell_ID"].values,
                                                         "cell": df["cell"].values})

adata = ad.AnnData(X=df_counts.values, obs=obs_metadata, var=var_metadata)

adata.obs["niche_type"] = df_metadata["spatialClusteringAssignments"].values
adata.obs["cell_type"] = df_metadata["RefinedClusts_Final"].values
adata.obs["cell_area"] = df_metadata["Area.um2"].values	

# Convert pixel value to micron
adata.obsm["spatial"] = np.array(
    df_metadata[["CenterX_global_px", "CenterY_global_px"]] * 0.12)

# Get ensembl ids
genes = adata.var.index.tolist()
ensembl_ids = []
for gene in genes:
    try:
        ensembl_ids.append(ensembl.gene_ids_of_gene_name(gene)[0])
    except ValueError as e:
        ensembl_ids.append(gene)
        print(f"Gene {gene} not found in Ensembl release.")
adata.var["ensembl_id"] = ensembl_ids

#### 2.1.2 QC Data

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1])


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2])

sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

#### 2.1.3 Store Data

In [ ]:
adata.write(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

#### 2.1.4 Explore Data

In [ ]:
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

In [ ]:
adata.X.toarray()

In [ ]:
# Compute PCA and UMAP
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color=["cell_type"])

In [ ]:
sc.pl.umap(adata, color=["cell_types"])

In [ ]:
# Validate counts
print((adata.X == adata.X.astype(np.int)).sum())
print(adata.X.shape[0] * adata.X.shape[1])

In [ ]:
# Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key="spatial",
                        radius=27.5,
                        )

# Add self loops to spatial neighborhood graph
adata.obsp["spatial_connectivities"].setdiag(1)
    
# Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

In [ ]:
np.mean(adata.obsp["spatial_connectivities"].sum(1))

##### 2.1.4.1 Overview

In [ ]:
# Check distribution of total counts across space
adata.obs["total_counts"] = adata.X.sum(1)
sq.pl.spatial_scatter(adata, color="total_counts", shape=None) 

In [ ]:
# Visualize example gene in tissue
# sq.pl.spatial_scatter(adata, color="MBP", shape=None)

In [ ]:
# Visualize cell types in tissue
sq.pl.spatial_scatter(adata, color="cell_type", shape=None) 

In [ ]:
# Visualize niche types in tissue
sq.pl.spatial_scatter(adata, color="niche_type", shape=None) 

##### 2.1.4.2 Raw Counts

In [ ]:
# Compute UMAP based on (raw) counts
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

In [ ]:
# Compute UMAP based on (raw) neighborhood counts
sc.pp.pca(adata, layer="X_neighborhood")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

##### 2.1.4.3 Read Depth Normalized Counts

In [ ]:
# Read depth normalize counts
adata.X = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000

In [ ]:
# Compute UMAP based on read depth normalized counts
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in read depth normalized count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in read depth normalized count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

In [ ]:
# Read depth normalize neighborhood counts
adata.layers["X_neighborhood"] = (
    adata.layers["X_neighborhood"] /
    adata.layers["X_neighborhood"].sum(1).reshape(-1, 1) *
    10000)

In [ ]:
# Compute UMAP based on read depth normalized neighborhood counts
sc.pp.pca(adata, layer="X_neighborhood")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in depth normalized neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in depth normalized neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

##### 2.1.4.4 Mean Reg Std Normalized Counts (Seurat V3)

In [ ]:
# Compute means and vars of read-depth normalized cell gene counts
cell_gene_means = np.mean(adata.X, axis=0)
cell_gene_vars = np.var(adata.X, axis=0)

# Compute means and vars of read-depth normalized neighborhood gene counts
neighborhood_gene_means = np.mean(adata.layers["X_neighborhood"], axis=0)
neighborhood_gene_vars = np.var(adata.layers["X_neighborhood"], axis=0)

# Learn mean-variance relationships for cell and neighborhood genes respectively
# to remove (only) expected variance based on mean to stabilize variances across genes.
# Inspired by seurat scaling to determine highly variable genes
# https://github.com/scverse/scanpy/blob/4642cf8e2e51b257371792cb4fcb9611c0a81123/scanpy/preprocessing/_highly_variable_genes.py#L26
not_const_cell_genes = cell_gene_vars > 0
cell_gene_est_log_vars = np.zeros(adata.shape[1], dtype=np.float64)
y = np.log10(cell_gene_vars[not_const_cell_genes])
x = np.log10(cell_gene_means[not_const_cell_genes])
cell_gene_model = loess(x, y, span=0.3, degree=2)
cell_gene_model.fit()
cell_gene_est_log_vars[not_const_cell_genes] = cell_gene_model.outputs.fitted_values
cell_gene_reg_stds = np.sqrt(10**cell_gene_est_log_vars)

not_const_neighborhood_genes = neighborhood_gene_vars > 0
neighborhood_gene_est_log_vars = np.zeros(adata.shape[1], dtype=np.float64)
y = np.log10(neighborhood_gene_vars[not_const_neighborhood_genes])
x = np.log10(neighborhood_gene_means[not_const_neighborhood_genes])
neighborhood_gene_model = loess(x, y, span=0.3, degree=2)
neighborhood_gene_model.fit()
neighborhood_gene_est_log_vars[not_const_neighborhood_genes] = neighborhood_gene_model.outputs.fitted_values
neighborhood_gene_reg_stds = np.sqrt(10**neighborhood_gene_est_log_vars)

# Normalize cell and neighborhood gene counts by mean and reg std
adata.X = (adata.X - cell_gene_means) / cell_gene_reg_stds
adata.layers["X_neighborhood"] = (
    adata.layers["X_neighborhood"] - neighborhood_gene_means) / neighborhood_gene_reg_stds

In [ ]:
# Compute UMAP based on mean std normalized counts
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in mean std normalized count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in mean std normalized count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

In [ ]:
# Compute UMAP based on mean std normalized neighborhood counts
sc.pp.pca(adata, layer="X_neighborhood")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in mean std normalized neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in mean std normalized neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

##### 2.1.4.5 Non-zero Median Division Normalized Counts (Geneformer)

In [ ]:
# Reverse mean and reg std normalization
adata.X = adata.X * cell_gene_reg_stds + cell_gene_means
adata.layers["X_neighborhood"] = (
    adata.layers["X_neighborhood"] * neighborhood_gene_reg_stds + neighborhood_gene_means)

# Compute non-zero medians
masked_data = np.ma.masked_equal(adata.X, 0)
cell_gene_nzmedians = np.ma.mean(masked_data, axis=0)
cell_gene_nzmedians = cell_gene_nzmedians.filled(0)
masked_data_neighborhood = np.ma.masked_equal(adata.layers["X_neighborhood"], 0)
neighborhood_gene_nzmedians = np.ma.mean(masked_data_neighborhood, axis=0)
neighborhood_gene_nzmedians = neighborhood_gene_nzmedians.filled(0)

# Perform non-zero median division normalization
adata.X = adata.X / cell_gene_nzmedians
adata.layers["X_neighborhood"] = adata.layers["X_neighborhood"] / neighborhood_gene_nzmedians

In [ ]:
# Compute UMAP based on non-zero median division normalized counts
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in non-zero median division normalized count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in non-zero median division normalized count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

In [ ]:
# Compute UMAP based on non-zero median division normalized neighborhood counts
sc.pp.pca(adata, layer="X_neighborhood")
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Visualize cell types in non-zero median division normalized neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
# Visualize niche types in non-zero median division normalized neighborhood-count-based UMAP space
sc.pl.umap(adata,
           color=["niche_type"])

### 2.2 Xenium Human Brain Cancer

- Data access: https://www.10xgenomics.com/datasets/ffpe-human-brain-cancer-data-with-human-immuno-oncology-profiling-panel-and-custom-add-on-1-standard

In [ ]:
dataset_name = "xenium_human_brain_cancer"
species = "homo_sapiens"

In [ ]:
ensembl = pyensembl.EnsemblRelease(111, species=species)

In [ ]:
# Read data
adata = sc.read_10x_h5(
    filename=f"{st_data_bronze_folder_path}/{dataset_name}/cell_feature_matrix.h5")
metadata = pd.read_csv(f"{st_data_bronze_folder_path}/{dataset_name}/cells.csv.gz")

In [ ]:
# Retrieve clusters
with tarfile.open(f"{st_data_bronze_folder_path}/{dataset_name}/analysis.tar.gz", 'r:gz') as tar:
    extract_path = f"{st_data_bronze_folder_path}/{dataset_name}"
    tar.extractall(path=extract_path)

clusters = pd.read_csv(f"{st_data_bronze_folder_path}/{dataset_name}/analysis/clustering/gene_expression_graphclust/clusters.csv")

#### 2.2.1 Preprocess Data

In [ ]:
# Add spatial coordinates and cluster labels
adata.obs = adata.obs.join(metadata.set_index("cell_id"))
adata.obs = adata.obs.join(clusters.set_index("Barcode"))
adata.obs.rename({"Cluster": "cluster"}, axis=1, inplace=True)

adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()

adata.var["ensembl_id"] = adata.var["gene_ids"]
adata.var = adata.var[["ensembl_id"]]

#### 2.2.2 QC Data

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1])


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2])

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3])

sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

#### 2.2.3 Store Data

In [ ]:
adata.write(f"{st_data_silver_folder_path}/xenium_human_brain_cancer/xenium_human_brain_cancer.h5ad")

#### 2.2.4 Explore Data

In [ ]:
# Check average number of neighbors
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key="spatial",
                        radius=27.50,
                        )

np.mean(adata.obsp["spatial_connectivities"].sum(1))

In [ ]:
# Check distribution of total counts across space
adata.obs["total_counts"] = adata.X.sum(1)
sq.pl.spatial_scatter(adata, color="total_counts", shape=None) 

### 2.3 Xenium Human Brain

In [ ]:
def load_s3_anndata_zarr(url='s3://hindlimb/datasets/20230412_multimodal/integrated/0.0.1/hindlimb-scrnaseq-integrated-anndata.zarr/',
                         endpoint="https://cog.sanger.ac.uk"):
    s3 = s3fs.S3FileSystem(anon=True, client_kwargs = {
        'endpoint_url':endpoint,
        'region_name':'us-east-1'
    })
    store=s3fs.S3Map(
        s3=s3,
        root=url.replace("https", "s3").replace(".cog.sanger.ac.uk", "")
    )
    return ad.read_zarr(store)

In [ ]:
# Preprocess files and store on disk as 'h5ad' files
with open(st_data_url_folder_path + "/xenium_human_gbm.txt", "r") as file:
    gbm_urls = file.readlines()
    for url in gbm_urls:
        adata = load_s3_anndata_zarr(url.strip())

        # Store 'ensembl_id' as expected by tokenizer
        adata.var["ensembl_id"] = adata.var["gene_ids"]
        
        # Convert spatial coordinates to microns
        adata.obsm["spatial"] = adata.obsm["X_spatial"] * adata.uns["xenium"]["pixel_size"]
        del(adata.obsm["X_spatial"])

        adata.write(f"{st_data_gold_folder_path}/xenium_human_gbm/{url.split('/')[-2].split('-anndata')[0]}.h5ad")
        del(adata)

### 2.4 MERFISH Mouse Hypothalamus

- Data access: http://sdmbench.drai.cn/ (https://datadryad.org/stash/dataset/doi:10.5061/dryad.8t8s248)
- "Expression values for the 135 genes measured in the combinatorial smFISH run were determined as the total counts per cell divided by the cell volume and scaled by 1000. Expression values for the 21 genes (including Fos) measured in the non-combinatorial, sequential FISH rounds were arbitrary fluorescence units per µm^3, but the same scale is used for all cells."

In [ ]:
dataset_name = "merfish_mouse_hypothalamus"
species = "mus_musculus"

In [ ]:
ensembl = pyensembl.EnsemblRelease(111, species=species)

#### 2.4.1 Preprocess & Store Data

In [ ]:
# Read data
file_list = [
    "MERFISH_0.04_20240609141556.h5ad",
    "MERFISH_0.09_20240609150351.h5ad",
    "MERFISH_0.14_20240609150353.h5ad",
    "MERFISH_0.19_20240609150355.h5ad",
    "MERFISH_0.24_20240609150356.h5ad"]

for i, file in enumerate(file_list):
    adata = sc.read_h5ad(f"{st_data_bronze_folder_path}/{dataset_name}/{file}")
    adata.obs["cell_type"] = adata.obs["cell_class"]
    del(adata.obs["cell_class"])
    adata.obs["niche_type"] = adata.obs["ground_truth"]
    del(adata.obs["ground_truth"])

    genes = [gene for gene in adata.var_names]
    
    # Rename some genes with names from Ensembl database
    for old_gene_name, new_gene_name in zip(["Cyr61"],
                                            ["Ccn1"]):
        idx = genes.index(old_gene_name)
        genes[idx] = new_gene_name

    adata.var_names = genes

    if i == 0:
        # Get ensembl ids
        genes = adata.var.index.tolist()
        ensembl_ids = []
        for gene in genes:
            try:
                ensembl_ids.append(ensembl.gene_ids_of_gene_name(gene)[0])
            except ValueError as e:
                ensembl_ids.append(gene)
                print(f"Gene {gene} not found in Ensembl release.")
    adata.var["ensembl_id"] = ensembl_ids

    os.makedirs(f"{st_data_silver_folder_path}/{dataset_name}", exist_ok=True)
    adata.write(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch{i+1}.h5ad")

#### 2.4.2 Explore Data

In [ ]:
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch1.h5ad")

In [ ]:
# Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key="spatial",
                        radius=27.5,
                        )

# Add self loops to spatial neighborhood graph
adata.obsp["spatial_connectivities"].setdiag(1)
    
# Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

##### 2.4.3.1 Overview

In [ ]:
# Check distribution of total counts across space
adata.obs["total_counts"] = adata.X.sum(1)
sq.pl.spatial_scatter(adata, color="total_counts", shape=None) 

In [ ]:
# Visualize example gene in tissue
# sq.pl.spatial_scatter(adata, color="Ace2", shape=None)

In [ ]:
# Visualize cell types in tissue
sq.pl.spatial_scatter(adata, color="cell_type", shape=None) 

In [ ]:
# Visualize niche types in tissue
sq.pl.spatial_scatter(adata, color="niche_type", shape=None) 

In [ ]:
np.mean(adata.obsp["spatial_connectivities"].sum(0))

### 2.5 Simulations

#### 2.5.1 Run Simulations

Run './data_simulation/data_simulation.ipynb' notebook.

#### 2.5.3 Preprocess & Store Data

In [ ]:
n_sim = 1
n_genes = 1105
n_locs = 10000

In [ ]:
sim1_name = f"sim{n_sim}_{n_genes}fts_{n_locs}locs"
os.makedirs(f"{st_data_bronze_folder_path}/{sim1_name}", exist_ok=True)

In [ ]:
for sim_name in [sim1_name]:

    adata = sc.read_h5ad(f"{st_data_bronze_folder_path}/{sim_name}/{sim_name}.h5ad")
    
    adata.obs["cell_types"] = adata.obs["cell_type"]
    del(adata.obs["cell_type"])
    adata.obs["niche_types"] = adata.obs["region_label"]
    del(adata.obs["region_label"])
    adata.obsm["spatial"] = np.array(adata.obs[["x", "y"]] * 0.35) # this is set to achieve an average number of neighbors of around 6 (similar to Nanostring CosMx Human Brain data)
    del(adata.obs["x"])
    del(adata.obs["y"])
    
    adata.var["ensembl_id"] = "ENS" + adata.var_names # this is added to make the tokenization work

    adata.X = sp.csr_matrix(adata.X)

    os.makedirs(f"{st_data_silver_folder_path}/{sim_name}", exist_ok=True)
    adata.write(f"{st_data_silver_folder_path}/{sim_name}/{sim_name}.h5ad")

In [ ]:
import numpy
np.arange(0.2, 0.5, 0.1)

#### 2.5.4 Explore Data

In [ ]:
n_sim = 1
n_genes = 1105
n_locs = 10000

sim_name = f"sim{n_sim}_{n_genes}fts_{n_locs}locs"
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{sim_name}/{sim_name}.h5ad")

In [ ]:
# Compute PCA and UMAP
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["cell_types"])

In [ ]:
specific_cell_types = ['Celltype2', 'Celltype5']
adata_subset = adata[adata.obs['cell_types'].isin(specific_cell_types), :]
sc.pl.umap(adata_subset, color=['cell_types'])

In [ ]:
sc.pl.umap(adata, color=["niche_types"])

In [ ]:
# Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key="spatial",
                        radius=27.5,
                        )

# Add self loops to spatial neighborhood graph
adata.obsp["spatial_connectivities"].setdiag(1)

# Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

In [ ]:
np.mean(adata.obsp["spatial_connectivities"].sum(0))

In [ ]:
# Visualize cell types in tissue
sq.pl.spatial_scatter(adata, color="cell_types", shape=None) 

In [ ]:
# Visualize region labels in tissue
sq.pl.spatial_scatter(adata, color="niche_types", shape=None) 

### 2.6 snRNA-seq Human MS Data (scGPT Benchmark)

Data access: https://scgpt.readthedocs.io/en/latest/tutorial_annotation.html

In [ ]:
dataset_name = "snrnaseq_human_ms"
species = "homo_sapiens"

#### 2.6.1 Preprocess & Store Data

This data is already normalized.

In [ ]:
adata = sc.read_h5ad(f"{sc_data_bronze_folder_path}/{dataset_name}/c_data.h5ad")
adata_test = sc.read_h5ad(f"{sc_data_bronze_folder_path}/{dataset_name}/filtered_ms_adata.h5ad")
adata.obs["cell_types"] = adata.obs["Factor Value[inferred cell type - authors labels]"].astype("category")
adata_test.obs["cell_types"] = adata_test.obs["Factor Value[inferred cell type - authors labels]"].astype("category")
adata.obs["batch_id"]  = adata.obs["str_batch"] = "0"
adata_test.obs["batch_id"]  = adata_test.obs["str_batch"] = "1"
adata.var["ensembl_id"] = adata.var["index_column"]
adata_test.var["ensembl_id"] = adata_test.var["index_column"]
adata.var.set_index(adata.var["gene_name"], inplace=True)
adata_test.var.set_index(adata.var["gene_name"], inplace=True)
adata = adata.concatenate(adata_test, batch_key="str_batch")
batch_id_labels = adata.obs["str_batch"].astype("category").cat.codes.values
adata.obs["batch_id"] = batch_id_labels
adata.obsm["spatial"] = np.arange(0, adata.shape[0]*2*1000, 1000).reshape(adata.shape[0], 2) # just to make it work with tokenizer
os.makedirs(f"{sc_data_silver_folder_path}/{dataset_name}", exist_ok=True)
adata.write(f"{sc_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

#### 2.6.2 Explore Data

In [ ]:
adata = sc.read_h5ad(f"{sc_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

In [ ]:
# Compute PCA and UMAP
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["cell_types"])

In [ ]:
sc.pl.umap(adata, color=["GFAP"]) # astrocyte marker gene (others: "ALDH1L1", "AQP4")

In [ ]:
sc.pl.umap(adata, color=["AQP4"]) 

In [ ]:
adata.var[adata.var["gene_name"].isin(["GFAP", "AQP4"])]

### 2.7 MERFISH Mouse Brain

Data access: https://cellxgene.cziscience.com/collections/0cca8620-8dee-45d0-aef5-23f032a5cf09 (8,380,288 cells)

In [ ]:
dataset_name = "merfish_mouse_brain"

#### 2.7.1 Preprocess & Store Data

In [ ]:
os.makedirs(f"{st_data_silver_folder_path}/{dataset_name}", exist_ok=True)

In [ ]:
batch_idx_increment = 1

for i, file in enumerate(
    ["animal1_coronal",
     "animal2_coronal",
     "animal3_sagittal",
     "animal4_sagittal"]):
    adata = sc.read_h5ad(f"{st_data_bronze_folder_path}/merfish_mouse_brain/WB_MERFISH_{file}.h5ad")

    sc.pp.filter_cells(adata, min_genes=10)

    label_encoder = LabelEncoder()

    adata.var["ensembl_id"] = adata.var.index
    adata.var.index = adata.var["gene_name"]
    adata.var = adata.var[["ensembl_id"]]
    adata.obsm["spatial"] = adata.obsm["X_spatial"]
    del(adata.obsm["X_spatial"])
    adata.X = adata.raw.X
    adata.obs = adata.obs[["donor_id", "brain_section_label", "major_brain_region", "ccf_region_name", "cell_type"]]
    adata.obs["batch"] = label_encoder.fit_transform(adata.obs['brain_section_label'])
    adata.obs["batch"] = adata.obs["batch"].apply(lambda x: "batch" + str(x + batch_idx_increment))

    for batch in adata.obs["batch"].unique():
        adata_batch = adata[adata.obs["batch"] == batch]
        adata_batch.write(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch{batch[5:]}.h5ad")
        
    batch_idx_increment += adata.obs["batch"].nunique()

#### 2.7.2 Explore Data

In [ ]:
n_cells = 0
for batch in range(1, 240):
    adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch{batch}.h5ad")
    # Validate counts
    print((adata.X == adata.X.astype(np.int64)).sum())
    print(adata.X.shape[0] * adata.X.shape[1])
    n_cells += len(adata)
    print(n_cells)

### 2.8 xhs1000-39b_1p (Annotated Xenium Human Skin Data - Collab with Lloyd Steele)

In [ ]:
# Define bronze config
dataset_name = 'xhs1000-39b_1p'
cell_type_key = 'Annotation'
niche_type_key = 'NICHE_NAMES'
batch_key = 'info_id'

In [ ]:
os.makedirs(f'{st_data_silver_folder_path}/{dataset_name}', exist_ok=True)

#### 2.8.1 Preprocess & Store Data

In [ ]:
# Read data
adata = sc.read_h5ad(f"{st_data_bronze_folder_path}/{dataset_name}/adata_5k_xenium_skin.h5ad")
print(adata)

# Store raw counts
adata.X = adata.layers["counts"]

In [ ]:
# Create batch mapping dict
batch_mapping_dict = {value: idx for idx, value in enumerate(adata.obs['info_id'].unique().tolist())}

for batch in adata.obs['info_id'].unique().tolist():
    adata_batch = adata[adata.obs['info_id'] == batch]
    adata_batch.obs['cell_id'] = [
        f'1000_{batch_idx}_{cell_idx}' for batch_idx, cell_idx in zip(
            [batch_mapping_dict[batch] for batch in adata_batch.obs['info_id'].values.tolist()], range(len(adata_batch.obs)))]
    adata_batch.obs['cell_type'] = adata_batch.obs[cell_type_key]
    adata_batch.obs['niche_type'] = adata_batch.obs[niche_type_key]
    adata_batch.obs = adata_batch.obs[['cell_id', 'cell_type', 'niche_type', 'Drug', 'Responder', 'Timepoint', 'baseline_postrx', 'Site_status']]
    adata_batch.write(f'{st_data_silver_folder_path}/{dataset_name}/adata_batch{batch_mapping_dict[batch]}.h5ad')

#### 2.8.2 Explore Data

In [ ]:
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/adata_batch0.h5ad")
print(adata)

In [ ]:
sq.pl.spatial_scatter(adata_batch1, color=cell_type_key, shape=None, figsize=(12, 12))   

In [ ]:
sq.pl.spatial_scatter(adata_batch1, color=niche_type_key, shape=None, figsize=(9.15, 9.15))   

### 2.9 xhb1001-8b_1p (Unannotated Xenium Human GBM Data - Collab with Bayraktar Lab)

In [ ]:
adata_test = sc.read_h5ad("/lustre/scratch126/cellgen/team361/DATASETS/LP_GBM/LP_GBM__AT15-BRA-4-FO-G1-S29.h5ad")

In [ ]:
adata_test.obs.columns

In [ ]:
adata.obsm["spatial"]

#### 2.9.1 Explore Data

In [ ]:
# Visualize cell types in tissue
sq.pl.spatial_scatter(adata, color="cell_types", shape=None) 

In [ ]:
sq.pl.spatial_scatter(adata, color=cell_type_key, shape=None, figsize=(12, 12))   

### 2.10 chl59-8b_1p

In [ ]:
dataset_name = 'chl59-8b_1p'

In [ ]:
# Load adata with NicheCompass annotations
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/chl59-8b_1p/nanostring_cosmx_human_nsclc_reference.h5ad')

# Define dict to map from NicheCompass batches to NicheJEPA batches
batch_mapping_dict = {
    'c_1': 'batch6', # reference
    'c_2': 'batch7', # reference
    'c_3': 'batch2', # reference
    'c_4': 'batch4', # reference
    'c_5': 'batch1', # reference
    'c_6': 'batch5', # reference
    'c_7': 'batch0', # reference
}

# Label niches (based on NicheCompass reproducibility repo)
leiden2niche = {
    '0': '1- Tumor (stroma border)', '2': '2- Tumor interior', '5': '3- Tumor (neutrophil border)', '7': '4- Tumor interior', '11': '5- Infiltrated tumor',
    '3': '6- Tumor-infiltrating neutrophils', '4': '7- Neutrophil expansion', '1': '8- Stroma', '6': '9- Plasmablast rich stroma', '9': '10- Lymphoid rich stroma', '8': '11- Lymphoid aggregates', '10': '12- Macrophage rich stroma'
}
adata.obs['niche'] = adata.obs['latent_leiden_0.45'].map(leiden2niche)
niche_colors = {
    leiden2niche.get(k, k): v
    for k, v in cluster_colors.items()
}

cluster_key = 'niche'
spot_size = 80

for batch in adata.obs['batch'].unique().tolist():
    adata_batch = adata[adata.obs['batch'] == batch]
    adata_batch.obs = adata_batch.obs[['cell_type', 'niche']]

    new_idx = [batch_mapping_dict[item[:3]] + item[3:] for item in adata_batch.obs.index.tolist()]
    adata_batch.obs_names = new_idx
    adata_batch.obs.to_csv(f'/lustre/scratch126/cellgen/team361/DATASETS/bronze/chl59-8b_1p/nichecompass_annotations_{batch}.csv', index=True)
    
    sc.pl.spatial(adata_batch,
                  color=cluster_key,
                  palette=niche_colors,
                  spot_size=spot_size,
                  show=False)
    plt.savefig(f"{artifacts_folder_path}/benchmarking/figures/{dataset_name}_{batch}.svg", format="svg", bbox_inches='tight')

### 2.11 chr78-11b_1p

In [ ]:
dataset_name = 'chr78-11b_1p'

In [ ]:
label2niche = {
    '0': 'Adrenal',
    '1': 'Background',
    '2': 'Cervix',
    '3': 'Ducts',
    '4': 'Epididymis',
    '5': 'Epoophron',
    '6': 'External Genitalia',
    '7': 'FallopianTube',
    '8': 'Genital Tubercle',
    '9': 'Gonad',
    '10': 'LabioScrotal Swelling',
    '11': 'LowerTract',
    '12': 'Nephros',
    '13': 'Testis',
    '14': 'Unassigned',
    '15': 'UpperVagina',
    '16': 'Urogenital Sinus',
    '17': 'UrogenitalSinus',
    '18': 'Uterus',
    '19': 'VaginalPlate',
    '20': 'Vas Deferens'
}

niche_colors = {
    label2niche.get(k, k): v
    for k, v in cluster_colors.items()
}

file_names = ['CM007_annotated.h5ad',
              'CM008_annotated.h5ad',
              'CM009_annotated.h5ad',
              'CM011_annotated.h5ad',
              'CM013_annotated.h5ad',
              'CM015_annotated.h5ad',
              'CM016_annotated.h5ad',
              'CM017_annotated.h5ad',
              'CM018_annotated.h5ad',
              'CM019_annotated.h5ad',
              'CM020_annotated.h5ad']

for file_name in file_names:
    adata_batch = sc.read_h5ad(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/chr78-11b_1p/{file_name}')

    cluster_key = 'broad_anatomy'
    spot_size = 240

    sc.pl.spatial(adata_batch,
                  color=cluster_key,
                  palette=niche_colors,
                  spot_size=spot_size,
                  show=False)
    plt.savefig(f"{artifacts_folder_path}/benchmarking/figures/{dataset_name}_{file_name.split('_')[0]}.svg", format="svg", bbox_inches='tight')

### 2.12 xhs49-1b_1p

In [ ]:
dataset_name = 'xhs49-1b_1p'

In [ ]:
# Load adata
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhs49-1b_1p/adata_xenium5k_melanoma_11jan.h5ad')

In [ ]:
label2niche = {
    '0': 'Epidemis',
    '1': 'Superficial dermis',
    '2': 'Melanoma+infiltrate',
    '3': 'Melanoma',
    '4': 'Immune infiltrate Thi',
    '5': 'Immune infiltrate Mac hi',
    '6': 'Macrophage_fat_region',
    '7': 'Melanoma+blood vessels'
}

niche_colors = {
    label2niche.get(k, k): v
    for k, v in cluster_colors.items()
}

cluster_key = 'niche_name'
spot_size = 20

sc.pl.spatial(adata,
              color=cluster_key,
              palette=niche_colors,
              spot_size=spot_size,
              show=False)
plt.savefig(f"{artifacts_folder_path}/benchmarking/figures/{dataset_name}.svg", format="svg", bbox_inches='tight')

In [ ]:
niche_colors

In [ ]:
0014285
0014285

In [ ]:
adata_temp = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhb1002-AT9-6b_1p/AT9-BRA-5-FO-4-S52_ii__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad')

In [ ]:
len(adata_temp.obs)

In [ ]:
adata_temp.obs.head()

In [ ]:
adata_temp.uns['slideID'] = '0014285_2'

In [ ]:
adata_temp.obs['sample_id'] = '0037751'

In [ ]:
adata_temp.obs.head()

In [ ]:
# adata_temp.write('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p/AT9-BRA-5-FO-4-S52_ii__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad')

In [ ]:
adata_temp.obs['sample_id'][0]

In [ ]:
import zipfile

raw_path = '/lustre/scratch126/cellgen/team361/DATASETS/bronze/chp60-1b_1p/Pancreas-CosMx-WTx-FlatFiles.zip'  # Replace with your actual path

with zipfile.ZipFile(raw_path, 'r') as zipp:
    for info in zipp.infolist():
        print(f"{info.filename}: Compression Type {info.compress_type}")

### 2.13 xho47-1b_1p

In [ ]:
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/xho47-1b_1p/Xenium_Prime_Human_Ovary_FF_outs.h5ad')

In [ ]:
adata1 = adata[adata.obsm['spatial'][:, 1] > ((np.max(adata.obsm['spatial'][:, 1]) - np.min(adata.obsm['spatial'][:, 1])) / 2)]
adata2 = adata[adata.obsm['spatial'][:, 1] < ((np.max(adata.obsm['spatial'][:, 1]) - np.min(adata.obsm['spatial'][:, 1])) / 2)]

In [ ]:
adata1.write('/lustre/scratch126/cellgen/team361/DATASETS/silver/xho47-1b_1p/Xenium_Prime_Human_Ovary_FF_outs_part1.h5ad')
adata2.write('/lustre/scratch126/cellgen/team361/DATASETS/silver/xho47-1b_1p/Xenium_Prime_Human_Ovary_FF_outs_part2.h5ad')

In [ ]:
sc.pl.spatial(adata1,
              spot_size=100)

In [ ]:
sc.pl.spatial(adata2,
              spot_size=100)

### 2.14 mhl80-7b_1p

In [ ]:
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/mhl80-8b_1p/adata_healthy_diseased_merfish.h5ad')

In [ ]:
adata.X = sp.csr_matrix(adata.X)

In [ ]:
adata.layers['counts'] = sp.csr_matrix(adata.raw.X.copy())

In [ ]:
adata.write('/lustre/scratch126/cellgen/team361/DATASETS/bronze/mhl80-8b_1p/adata_healthy_diseased_merfish.h5ad')

In [ ]:
adata.X = adata.X.todense()

In [ ]:
print(adata.X)

In [ ]:
def overwrite(adata):
    adata.X = sp.csr_matrix(adata.X)
    print(adata.X)

In [ ]:
overwrite(adata)

### Human Cohort 1 50m

In [ ]:
# Load token dict and get token dict-specfic params
with open('/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/human_cohort1_50m/token_dictionary.pkl', 'rb') as file:
    token_dict = pickle.load(file)

In [ ]:
# Check token dict
pp = pprint.PrettyPrinter(width=200)
pp.pprint(token_dict)

In [ ]:
# Define the old and new key names
old_key = '<dummy_batch_31>'
new_key = 'spv_1007_batch7'

# Overwrite old key with new key
if old_key in token_dict:
    token_dict[new_key] = token_dict.pop(old_key)
    print(f"Key '{old_key}' has been replaced with '{new_key}'.")
else:
    print(f"Key '{old_key}' does not exist in the dictionary.")

In [ ]:
# Write to disk
with open('/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/human_cohort1_50m/token_dictionary.pkl', 'wb') as file:
    pickle.dump(token_dict, file)

In [ ]:
adata_test = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhc1001-1b_1p/NicheCompass_hs_XeniumAdultColon_AM_07112024_132853_raw.h5ad')

In [ ]:
adata_test.obsm['spatial']

In [ ]:
adata_test.obs.head()

In [ ]:
adata = adata_test

In [ ]:
adata_test.X.shape

In [ ]:
basis_target_read_depth = 153.48
target_read_depth_per_gene = 0.05

In [ ]:
X = adata.X / adata.X.sum(axis=1).reshape(-1, 1) * (
    basis_target_read_depth + adata.X.shape[1] * target_read_depth_per_gene)
X_dense = X.toarray()
nonzero_mask = X_dense > 0
nonzero_counts = np.sum(nonzero_mask, axis=0)
nonzero_sums = np.sum(X_dense * nonzero_mask, axis=0)
nonzero_means = np.true_divide(nonzero_sums, nonzero_counts)
means = X_dense.mean(0)

norm_factor_dict = {}
for idx, gene_id in enumerate(adata.var['ensembl_id'].values):
    if gene_id in norm_factor_dict.keys():
        norm_factor_dict[f"{gene_id}_nonzero_means"].append(nonzero_means[idx])
        norm_factor_dict[f"{gene_id}_means"].append(means[idx])
        norm_factor_dict[f"{gene_id}_n_cells"].append(len(data))
    else:
        norm_factor_dict[f"{gene_id}_nonzero_means"] = [nonzero_means[idx]]
        norm_factor_dict[f"{gene_id}_means"] = [means[idx]]
        norm_factor_dict[f"{gene_id}_n_cells"] = [len(adata)]

In [ ]:
norm_factor_dict

In [ ]:
adata.var['ensembl_id'].values

In [ ]:
nonzero_means

In [ ]:
count_means

In [ ]:
adata_test.uns['batch'] # 1, 4, 7, 9, 3, 5, 2, 6

In [ ]:
adata_test.obs.tail()

In [ ]:
adata_test.obs['_scvi_batch'].unique()

In [ ]:
adata_test.obsm['spatial']

In [ ]:
sc.settings.set_figure_params(
    figsize=(60, 36),
)

In [148]:
# Load token dict and get token dict-specfic params
with open('/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/artifacts/data/human_cohort1_60m_batchwise_norm_factors.pkl', 'rb') as file:
    batchwise_norm_factor_dict = pickle.load(file)

In [ ]:
gene_ids = list(set([key.split('_')[0] for key in norm_factors_dict.keys()]))

In [ ]:
len(gene_ids)

In [159]:
norm_factor_dict = {}

for gene_id in gene_ids:
    norm_factor_dict[gene_id] = {}
    norm_factor_dict[gene_id]["nonzero_mean"] = np.dot(
        np.array(batchwise_norm_factor_dict[f"{gene_id}_nonzero_means"]),
        np.array(batchwise_norm_factor_dict[f"{gene_id}_n_cells"])
    ) / np.array(batchwise_norm_factor_dict[f"{gene_id}_n_cells"]).sum()
    norm_factor_dict[gene_id]["mean"] = np.dot(
        np.array(batchwise_norm_factor_dict[f"{gene_id}_means"]),
        np.array(batchwise_norm_factor_dict[f"{gene_id}_n_cells"])
    ) / np.array(batchwise_norm_factor_dict[f"{gene_id}_n_cells"]).sum()
    norm_factor_dict[gene_id]["n_cells"] = np.array(batchwise_norm_factor_dict[f"{gene_id}_n_cells"]).sum()

norm_factor_df = pd.DataFrame.from_dict(norm_factor_dict, orient='index').reset_index()
norm_factor_df = norm_factor_df.rename(columns={'index': 'gene_id'})

In [ ]:
parsed_data = {}
for key, value in norm_factor_dict.items():
    gene_id, metric = key.split('_', 1)
    if gene_id not in parsed_data:
        parsed_data[gene_id] = {}
    
    parsed_data[gene_id][metric] = value

# Convert the parsed dictionary to a DataFrame
df = pd.DataFrame.from_dict(parsed_data, orient='index').reset_index()

# Rename the 'index' column to 'gene_id'
df = df.rename(columns={'index': 'gene_id'})

# Reorder columns if necessary
desired_order = ['gene_id', 'nonzero_mean', 'mean', 'n_cells']
df = df[desired_order]

In [154]:
with open(f'/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/artifacts/data/human_cohort1_60m_norm_factors.pkl', 'wb') as file:
    pickle.dump(norm_factor_dict, file)

In [155]:
norm_factor_dict

{'ENSG00000161911_nonzero_mean': 0.9368816270058502,
 'ENSG00000161911_mean': 0.030360260468743544,
 'ENSG00000161911_n_cells': 237620,
 'ENSG00000184371_nonzero_mean': 1.4044747757835205,
 'ENSG00000184371_mean': 0.32792640995012723,
 'ENSG00000184371_n_cells': 29378271,
 'ENSG00000178096_nonzero_mean': 1.1960843453492527,
 'ENSG00000178096_mean': 0.014396813655346176,
 'ENSG00000178096_n_cells': 48934,
 'ENSG00000135678_nonzero_mean': 1.1499752720572747,
 'ENSG00000135678_mean': 0.026984911367722162,
 'ENSG00000135678_n_cells': 65113,
 'ENSG00000136159_nonzero_mean': 0.7780270259446573,
 'ENSG00000136159_mean': 0.04335373365063986,
 'ENSG00000136159_n_cells': 237620,
 'ENSG00000242574_nonzero_mean': 1.7504215186516439,
 'ENSG00000242574_mean': 0.35729045649725855,
 'ENSG00000242574_n_cells': 9809827,
 'ENSG00000112164_nonzero_mean': 1.572342381174383,
 'ENSG00000112164_mean': 0.0030897491215892544,
 'ENSG00000112164_n_cells': 8375746,
 'ENSG00000185324_nonzero_mean': 1.34158063650687

13661254

In [139]:
norm_factors_dict[f"{gene_id}_nonzero_means"]

[3.6487825015271866,
 3.539272755183419,
 2.904315174949235,
 2.8710045889217324,
 3.501937715662859,
 3.4767514913936517,
 2.841973899020675,
 4.989737318532,
 3.2142798329495985,
 7.602517441281306,
 1.79548671233304,
 3.6414274664174284,
 1.3059439528477648,
 3.680304426768214,
 1.8828549609279184,
 2.1076672160187244,
 2.384848900326626,
 1.2110744090307033,
 2.5862258894001062,
 1.5836979609070436,
 2.2813385526302334,
 1.4015341381731845,
 1.6558018102168304,
 1.4237267778047509,
 2.5069452313646416,
 3.004464006329673,
 1.9196919040500866,
 1.8846939237153062,
 1.8194829900337846,
 2.022360098158686,
 0.9797402482280015,
 2.2293314690201296,
 2.5926360005390916,
 1.5102753051569835,
 1.9980021087751068,
 2.6613268196678397,
 2.9995329154644184,
 2.6174539704449336,
 3.465979635915512,
 1.8364275325319077,
 1.6430277000820945,
 1.9155690028422137,
 1.5351320488305935,
 2.109729958910833,
 2.385867631137216,
 2.1863633412963006,
 1.4881053203424375,
 3.0732563086502482,
 3.9066535

In [137]:
norm_factors_dict[f"{gene_id}_n_cells"]

[852832,
 860604,
 364148,
 574626,
 353895,
 570485,
 571988,
 272341,
 576336,
 777955,
 704966,
 184400,
 109208,
 397231,
 6195,
 18105,
 8532,
 15189,
 8546,
 32573,
 14663,
 10187,
 12702,
 15147,
 20287,
 9230,
 17384,
 10270,
 12014,
 14354,
 12297,
 6927,
 10922,
 16588,
 17832,
 2351,
 17926,
 11551,
 5706,
 7669,
 14367,
 10500,
 13487,
 14202,
 13548,
 14694,
 15566,
 18439,
 3885,
 14999,
 9062,
 9017,
 32145,
 140368,
 481485,
 460804,
 242436,
 360208,
 261495,
 230320,
 227444,
 188686,
 48934,
 2498,
 4487,
 4554,
 4640,
 500753,
 538573,
 500712,
 656036,
 272074,
 569435,
 275239]

In [ ]:
norm_factors_dict

In [ ]:
adata_test.layers['counts'] = adata_test.raw.X

In [ ]:
adata_test.obs['batch'].unique()

In [ ]:
adata_test = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT6.1-2b_1p/AT6-BRA-3-FO-4-S50-A1__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad')

In [ ]:
adata_test.var

In [ ]:
adata_test.obs.head()

In [ ]:
adata_test.obs['Donor_ID'].unique()

In [ ]:
adata.obs.tail()['cell']

In [ ]:
adata.obs['fov'].unique()